In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, Resized,
    RandFlipd, RandRotate90d, RandGaussianNoised, EnsureTyped,
)
from sklearn.model_selection import train_test_split
import pandas as pd
from monai.data import DataLoader
import torch
from monai.networks.nets import ViT
import pydicom
pydicom.config.settings.reading_validation_mode = pydicom.config.WARN


In [17]:
df_data = pd.read_csv("./data/labels_duke.csv")
data = df_data.to_dict(orient="records")


train_data, val_data = train_test_split(
    data, test_size=0.2, stratify=[d["label"] for d in data], random_state=42
)

In [ ]:

train_transforms = Compose([
    LoadImaged(keys="image", reader="PydicomReader"),          
    EnsureChannelFirstd(keys="image"),                      
    Orientationd(keys="image", axcodes="RAS"),              
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0),       
             mode="bilinear"),
    ScaleIntensityRangePercentilesd(                  
        keys="image", lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),      
    RandFlipd(keys="image", prob=0.5, spatial_axis=0),     
    RandRotate90d(keys="image", prob=0.5, spatial_axes=(0, 1)),
    RandGaussianNoised(keys="image", prob=0.2),
    EnsureTyped(keys="image"),
])

val_transforms = Compose([                                  
    LoadImaged(keys="image", reader="PydicomReader"),
    EnsureChannelFirstd(keys="image"),
    Orientationd(keys="image", axcodes="RAS"),
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99,
                                    b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),
    EnsureTyped(keys="image"),
])


/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [19]:
from monai.data import CacheDataset
import torch.multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

train_ds = CacheDataset(data=train_data, transform=train_transforms, cache_rate=1.0, num_workers=12)
val_ds   = CacheDataset(data=val_data,   transform=val_transforms,   cache_rate=1.0, num_workers=8)




Loading dataset: 100%|██████████| 188/188 [02:23<00:00,  1.31it/s]


In [ ]:
train_loader = DataLoader(train_ds, batch_size=30, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=30, num_workers=0)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ViT(
    in_channels=1,
    img_size=(96, 96, 96),
    patch_size=(16, 16, 16),
    classification=True,
    num_classes=5,
    post_activation=None,      
    save_attn=True
).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)


In [ ]:
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        x = batch["image"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(x)
        if isinstance(logits, tuple):   
            logits = logits[0]
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_loader:
            x = batch["image"].to(device)
            y = batch["label"].to(device)
            logits = model(x)
            if isinstance(logits, tuple):
                logits = logits[0]
            correct += (logits.argmax(1) == y).sum().item()
            total += y.numel()
    print(f"epoch {epoch}: val acc = {correct/total:.3f}")

epoch 0: val acc = 0.441
epoch 1: val acc = 0.426
epoch 2: val acc = 0.441
epoch 3: val acc = 0.441
epoch 4: val acc = 0.426
epoch 5: val acc = 0.441
epoch 6: val acc = 0.441
epoch 7: val acc = 0.426
epoch 8: val acc = 0.441
epoch 9: val acc = 0.441
epoch 10: val acc = 0.426
epoch 11: val acc = 0.431
epoch 12: val acc = 0.426
epoch 13: val acc = 0.426
epoch 14: val acc = 0.441
epoch 15: val acc = 0.495
epoch 16: val acc = 0.426
epoch 17: val acc = 0.426
epoch 18: val acc = 0.426
epoch 19: val acc = 0.426
epoch 20: val acc = 0.441
epoch 21: val acc = 0.426
epoch 22: val acc = 0.441
epoch 23: val acc = 0.500
epoch 24: val acc = 0.500
epoch 25: val acc = 0.495
epoch 26: val acc = 0.441
epoch 27: val acc = 0.426
epoch 28: val acc = 0.426
epoch 29: val acc = 0.426
epoch 30: val acc = 0.463
epoch 31: val acc = 0.441
epoch 32: val acc = 0.441
epoch 33: val acc = 0.426
epoch 34: val acc = 0.431
epoch 35: val acc = 0.441
epoch 36: val acc = 0.441
epoch 37: val acc = 0.441
epoch 38: val acc = 0.